In [1]:
print("llm 调用 tools,让llm 和外界交流")

llm 调用 tools,让llm 和外界交流


In [6]:
# npm install 
# 发起请求
!pip install requests

!pip install openai

Looking in indexes: https://mirrors.cloud.aliyuncs.com/pypi/simple
DEPRECATION: pytorch-lightning 1.7.7 has a non-standard dependency specifier torch>=1.9.*. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 23.3.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Looking in indexes: https://mirrors.cloud.aliyuncs.com/pypi/simple
DEPRECATION: pytorch-lightning 1.7.7 has a non-standard dependency specifier torch>=1.9.*. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/is

In [11]:
import requests
# python 类型约定，
# js 弱类型语言 
# -> str 返回值的类型
def get_weather(location: str) -> str:
    url = "https://api.seniverse.com/v3/weather/now.json"
    params = {
        "key": "SaVSOt7sYbwpka9iv",
        "location": location,
        "language": "zh-Hans"
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        data = resp.json()
        print(data)
        if "results" in data:
            r = data["results"][0]
            city = r["location"]["name"]
            # 当前天气对象
            now = r["now"]
            text = now["text"]
            temp = now["temperature"]
            #python 擅长机器学习 和字符处理
            return f"{city}当前天气：{text}, 气温 {temp}度"
        else:
            return "查询失败"
    except Exception as e:
        return  f"异常：{e}"
    
print(get_weather("抚州"))

{'results': [{'location': {'id': 'WSFXR95RZD21', 'name': '抚州', 'country': 'CN', 'path': '抚州,抚州,江西,中国', 'timezone': 'Asia/Shanghai', 'timezone_offset': '+08:00'}, 'now': {'text': '阴', 'code': '9', 'temperature': '8'}, 'last_update': '2025-11-18T20:37:05+08:00'}]}
抚州当前天气：阴, 气温 8度


In [12]:
# 用户user向llm 提问 自然语言
# 实时性的， 工具类的， llm 无法回答
# llm和原有的互联网文明桥接起来？ 智能互联网来了
# tools 来自openai 接口定义， 
from openai import OpenAI

client = OpenAI(
    api_key = 'sk-b96af4b3f3f7406ca872a01ee7276f95',
    base_url = 'https://api.deepseek.com/v1'
)

In [13]:
# openai 接口定义
tools = [
    # JSON 定义
    {
        # 一个工具就是一个函数
        "type": "function",
        # 函数的申明
        "function": {
            "name": "get_weather",
            "description": "获取指定城市的当前天气",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        # 北京天气怎么样？提取出来北京
                        "description": "城市名称，如'北京'"
                    }
                },
                "required": ["location"]
            }
        }
    }
]

In [17]:
import json
messages = [{"role": "user", "content": "北京天气怎么样"}]
response = client.chat.completions.create(
    model = 'deepseek-reasoner',
    messages = messages,
    tools = tools,
    tool_choice = "auto",
    # 生成的随机度
    temperature = 0.3
)
response_message = response.choices[0].message
# 多轮会话
print(response_message)
# tool_calls ChatCompletionMessageFunctionToolCall
messages.append(response_message)
if response_message.tool_calls:
    for tool_call in response_message.tool_calls:
        function_name = tool_call.function.name
        # json 字符串变成json 对象
        function_args = json.loads(tool_call.function.arguments)
        if function_name == "get_weather":
            function_response = get_weather(function_args["location"])
        else:
            function_response = "未知工具"
        messages.append({
            "tool_call_id": tool_call.id,
            "role": "tool",
            "name": function_name,
            "content": function_response
        })
else:
    print(response_message.content)

final_response = client.chat.completions.create(
    model="deepseek-reasoner",
    messages=messages,
    temperature=0.3
)
print(final_response.choices[0].message.content)

ChatCompletionMessage(content='我来帮您查询北京的天气情况。', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_8GOm91zqN5qQjlEw9ghkxbzd', function=Function(arguments='{"location": "北京"}', name='get_weather'), type='function', index=0)])
{'results': [{'location': {'id': 'WX4FBXXFKE4F', 'name': '北京', 'country': 'CN', 'path': '北京,北京,中国', 'timezone': 'Asia/Shanghai', 'timezone_offset': '+08:00'}, 'now': {'text': '晴', 'code': '1', 'temperature': '3'}, 'last_update': '2025-11-18T21:27:31+08:00'}]}
根据查询结果，北京当前天气情况：

- **天气状况**：晴 ☀️
- **当前气温**：3°C

今天北京天气不错，阳光明媚，但气温较低，建议您：
- 外出时注意保暖，穿着厚外套
- 由于天气晴朗，适合户外活动
- 早晚温差可能较大，注意适时增减衣物

需要了解未来几天的天气预报吗？我可以为您查询更详细的天气信息。
